In [34]:
# 全歯ある train データを探して抜きとる
import csv
import os
import numpy as np
import tifffile

def inport_train_data(INPUT_DIR, TARGET_SHAPE):
    CSV_PATH = os.path.join(INPUT_DIR, "dataset_info.csv")

    matched_indices = []
    with open(CSV_PATH, mode="r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("vol_shape") == TARGET_SHAPE:
                idx_str = (row.get("index") or "").strip()
                if idx_str != "":
                    matched_indices.append(int(idx_str))

    print(f"vol_shape={TARGET_SHAPE} の index 件数: {len(matched_indices)}")
    print("index一覧:", matched_indices)
    return matched_indices

def numpy_to_tiff(numpy_array, output_path):
    # numpy 配列を TIFF 形式で保存する
    tifffile.imwrite(output_path, numpy_array.astype(np.uint16))

if __name__ == "__main__":
    INPUT_DIR    = r"I:\TrainData"  # データセットのディレクトリを指定してください
    TARGET_SHAPE = "(804, 801, 801)"  # 探したい vol_shape を指定してください
    matched_indices = inport_train_data(INPUT_DIR, TARGET_SHAPE)
    index = 7
    label_numpy_path = os.path.join(INPUT_DIR, f"{index:03d}", f"label.npy")
    label_output_path = os.path.join(INPUT_DIR, f"{index:03d}", f"label.tif")
    numpy_array = np.load(label_numpy_path)
    numpy_to_tiff(numpy_array, label_output_path)

vol_shape=(804, 801, 801) の index 件数: 23
index一覧: [7, 9, 13, 18, 20, 22, 26, 29, 32, 41, 42, 44, 48, 51, 58, 60, 62, 64, 65, 80, 89, 90, 91]


In [35]:
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi
import os
import tifffile
import numpy as np

# label.tif を読み込み（セル1で作成したパスを優先）
if "label_output_path" in globals():
    label_tif_path = label_output_path
else:
    label_tif_path = os.path.join(INPUT_DIR, f"{index:03d}", "label.tif")

label_volume = tifffile.imread(label_tif_path)

# 大津の二値化
th = threshold_otsu(label_volume)
mask_otsu = (label_volume <= th) & (label_volume > 0)  # 0 は背景として除外

# なめらか化パラメータ（必要に応じて調整）
EROSION_ITER = 1
GAUSSIAN_SIGMA = 1.0
SMOOTH_THRESHOLD = 0.5

# 1) スムージング（ガウシアン）
mask_smooth = ndi.gaussian_filter(mask_otsu.astype(np.float32), sigma=GAUSSIAN_SIGMA)

# 2) 再2値化
mask_after_smooth = mask_smooth >= SMOOTH_THRESHOLD

# 3) エロージョン
mask_eroded = ndi.binary_erosion(mask_after_smooth, iterations=EROSION_ITER)

# 保存
preprocess_output_path = label_tif_path.replace(".tif", "_otsu_smooth_erode.tif")
tifffile.imwrite(preprocess_output_path, mask_eroded.astype(np.uint8) * 255)

print(f"Input label: {label_tif_path}")
print(f"Otsu threshold: {th}")
print(
    f"Saved: {preprocess_output_path} | sigma={GAUSSIAN_SIGMA}, "
    f"smooth_th={SMOOTH_THRESHOLD}, erosion_iter={EROSION_ITER}"
)


Input label: I:\TrainData\007\label.tif
Otsu threshold: 12484
Saved: I:\TrainData\007\label_otsu_smooth_erode.tif | sigma=1.0, smooth_th=0.5, erosion_iter=1


In [37]:
import os
import sys
import numpy as np
import tifffile
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi

# Watershed/src を import 可能にする
repo_root = r"D:\_study\ImageProcessing\study"
watershed_src = os.path.join(repo_root, "Watershed", "src")
if watershed_src not in sys.path:
    sys.path.append(watershed_src)

from FillHoles import fill_holes_in_binary_volume
from RodriguesRotation import rotate_volume_to_occlusal_and_left_right_parallel

# なめらか化パラメータ（必要に応じて調整）
EROSION_ITER = 1
GAUSSIAN_SIGMA = 1.0
SMOOTH_THRESHOLD = 0.5

if "matched_indices" not in globals() or len(matched_indices) == 0:
    raise ValueError("matched_indices が空です。先にセル1を実行してください。")

print(f"Processing indices: {matched_indices}")
done_indices = []

for index in matched_indices:
    index_dir = os.path.join(INPUT_DIR, f"{index:03d}")
    label_numpy_path = os.path.join(index_dir, "label.npy")
    label_tif_path = os.path.join(index_dir, "label.tif")

    # label.tif がなければ label.npy から作成
    if not os.path.exists(label_tif_path):
        if not os.path.exists(label_numpy_path):
            print(f"[SKIP] {index:03d}: label.npy がありません")
            continue
        numpy_array = np.load(label_numpy_path)
        tifffile.imwrite(label_tif_path, numpy_array.astype(np.uint16))

    # 1) 大津の二値化
    label_volume = tifffile.imread(label_tif_path)
    th = threshold_otsu(label_volume)
    mask_otsu = (label_volume <= th) & (label_volume > 0)

    # 2) スムージング -> 再二値化
    mask_smooth = ndi.gaussian_filter(mask_otsu.astype(np.float32), sigma=GAUSSIAN_SIGMA)
    mask_after_smooth = mask_smooth >= SMOOTH_THRESHOLD

    # 3) エロージョン
    mask_eroded = ndi.binary_erosion(mask_after_smooth, iterations=EROSION_ITER)

    preprocess_output_path = os.path.join(index_dir, "label_otsu_smooth_erode.tif")
    tifffile.imwrite(preprocess_output_path, mask_eroded.astype(np.uint8) * 255)

    # 4) FillHoles
    fill_output_path = os.path.join(index_dir, "label_otsu_smooth_erode_tmp_filled.tif")
    fill_holes_in_binary_volume(
        input_path=preprocess_output_path,
        output_path=fill_output_path,
        distance_output_path=None,
        per_slice=True,
        min_hole_size=None,
    )

    # 5) RodriguesRotation
    rot_input = tifffile.imread(fill_output_path)
    rotated, _ = rotate_volume_to_occlusal_and_left_right_parallel(
        volume=rot_input,
        spacing_zyx=(1.0, 1.0, 1.0),
        percentile=95.0,
        side="auto",
        pad=20,
        inplane_k45=4,
        first_side="low",
    )

    # 最終成果物のみ保存
    final_output_path = os.path.join(index_dir, "label_editted.tif")
    tifffile.imwrite(final_output_path, rotated.astype(np.uint8))

    # 中間ファイルを削除
    cleanup_paths = [
        preprocess_output_path,
        fill_output_path,
        os.path.join(index_dir, "label_otsu_smooth.tif"),
        os.path.join(index_dir, "label_otsu_smooth_erode_filled.tif"),
        os.path.join(index_dir, "label_otsu_smooth_erode_filled_aligned.tif"),
        os.path.join(index_dir, "label_otsu_smooth_erode_filled_aligned_info.json"),
    ]
    for file_path in cleanup_paths:
        if os.path.exists(file_path):
            os.remove(file_path)

    done_indices.append(index)
    print(f"[DONE] {index:03d} -> {final_output_path}")

print(f"Completed: {len(done_indices)} / {len(matched_indices)}")

Processing indices: [7, 9, 13, 18, 20, 22, 26, 29, 32, 41, 42, 44, 48, 51, 58, 60, 62, 64, 65, 80, 89, 90, 91]
Input : I:\TrainData\007\label_otsu_smooth_erode.tif
Output: I:\TrainData\007\label_otsu_smooth_erode_tmp_filled.tif
Filled voxels: 3188764085.0
[DONE] 007 -> I:\TrainData\007\label_editted.tif
Input : I:\TrainData\009\label_otsu_smooth_erode.tif
Output: I:\TrainData\009\label_otsu_smooth_erode_tmp_filled.tif
Filled voxels: 2587945893.0
[DONE] 009 -> I:\TrainData\009\label_editted.tif
Input : I:\TrainData\013\label_otsu_smooth_erode.tif
Output: I:\TrainData\013\label_otsu_smooth_erode_tmp_filled.tif
Filled voxels: 2172230907.0
[DONE] 013 -> I:\TrainData\013\label_editted.tif
Input : I:\TrainData\018\label_otsu_smooth_erode.tif
Output: I:\TrainData\018\label_otsu_smooth_erode_tmp_filled.tif
Filled voxels: 2856718232.0
[DONE] 018 -> I:\TrainData\018\label_editted.tif
Input : I:\TrainData\020\label_otsu_smooth_erode.tif
Output: I:\TrainData\020\label_otsu_smooth_erode_tmp_filled.